# MiBici Guadalajara — Data Cleaning & Quality Checks

This notebook reads the raw staging tables (all 12 months + stations) and applies validation and cleaning rules, documenting every decision. Output feeds into the star schema built in `03_modeling.ipynb`.

## 1. Setup: connect to PostgreSQL

Same credential pattern as the ingestion notebook. Reads from `.env`, nothing hardcoded.

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

## 2. Load all 12 months from staging into a single DataFrame

Rather than cleaning each month separately, combining them first means every check (duplicates, referential integrity, etc.) is evaluated across the full year at once

In [3]:
# Connection test
with engine.connect() as conn:
    tables = conn.execute(text(
        "SELECT table_name FROM information_schema.tables "
        "WHERE table_schema = 'staging' AND table_name LIKE 'trips_%' "
        "ORDER BY table_name;"
    )).fetchall()

trip_tables = [row[0] for row in tables]
print(trip_tables)

['trips_2025_01', 'trips_2025_02', 'trips_2025_03', 'trips_2025_04', 'trips_2025_05', 'trips_2025_06', 'trips_2025_07', 'trips_2025_08', 'trips_2025_09', 'trips_2025_10', 'trips_2025_11', 'trips_2025_12']


In [5]:
trip_dfs = []

for table in trip_tables:
    df = pd.read_sql(f"SELECT * FROM staging.{table}", con=engine)
    trip_dfs.append(df)

trips = pd.concat(trip_dfs, ignore_index=True)

print(trips.shape)
trips.head()

(4532032, 8)


,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
0,37162342,601273,M,1982.0,2025-01-01 00:00:44,2025-01-01 00:11:29,211,395
1,37162343,1571524,M,2003.0,2025-01-01 00:04:11,2025-01-01 00:15:07,35,260
2,37162344,1435200,F,1984.0,2025-01-01 00:05:27,2025-01-01 00:15:23,35,260
3,37162345,126327,M,1988.0,2025-01-01 00:07:47,2025-01-01 00:17:42,190,11
4,37162346,416121,M,1994.0,2025-01-01 00:08:52,2025-01-01 00:18:10,273,8


## 3. Load stations

Loaded separately since it's not a monthly series, it's the dimension table trips will eventually reference.

In [7]:
stations = pd.read_sql("SELECT * FROM staging.stations", con=engine)

print(stations.shape)
stations.head()

(484, 5)


,id,name,latitude,longitude,dpcapacity
0,2,(GDL-001) C. Epigmenio Glez./ Av. 16 de Sept.,20.666378,-103.348820,15
1,3,(GDL-002) C. Colonias / Av. Niños héroes,20.667228,-103.366000,15
2,4,(GDL-003) C. Vidrio / Av. Chapultepec,20.667690,-103.368252,19
3,5,(GDL-004) C. Ghilardi /C. Miraflores,20.691847,-103.362549,19
4,6,(GDL-005) C. San Diego /Calzada Independencia,20.681158,-103.339363,11


## 4. Parse datetime columns

`Inicio_del_viaje` and `Fin_del_viaje` came in from staging as plain text (staging preserves the raw source exactly). Converting them to proper datetime types here enables duration calculations and time-based analysis.

In [9]:
trips["Inicio_del_viaje"] = pd.to_datetime(trips["Inicio_del_viaje"], errors="coerce")
trips["Fin_del_viaje"] = pd.to_datetime(trips["Fin_del_viaje"], errors="coerce")

print(trips[["Inicio_del_viaje", "Fin_del_viaje"]].dtypes)
print(trips["Inicio_del_viaje"].isna().sum(), "unparseable start times")
print(trips["Fin_del_viaje"].isna().sum(), "unparseable end times")

Inicio_del_viaje    datetime64[ns]
Fin_del_viaje       datetime64[ns]
dtype: object
0 unparseable start times
0 unparseable end times


## 5. Calculate trip duration and flag invalid trips

Duration is derived from the two timestamp columns. Trips with zero or negative duration indicate data quality issues (e.g., a docking error rather than a real ride) and are flagged for review before deciding whether to exclude them.

In [11]:
trips["Duracion_min"] = (trips["Fin_del_viaje"] - trips["Inicio_del_viaje"]).dt.total_seconds() / 60

print(trips["Duracion_min"].describe())

count    4.532032e+06
mean     1.115420e+01
std      1.031854e+01
min      0.000000e+00
25%      5.883333e+00
50%      9.616667e+00
75%      1.495000e+01
max      1.150077e+04
Name: Duracion_min, dtype: float64


## 6. Investigate duration outliers before setting cutoffs

Rather than picking an arbitrary threshold, checking how many trips fall into suspicious ranges at each end of the distribution this informs what cutoff is actually defensible.

In [13]:
print("Trips with 0 minutes duration:", (trips["Duracion_min"] == 0).sum())
print("Trips under 1 minute:", (trips["Duracion_min"] < 1).sum())
print("Trips under 2 minutes:", (trips["Duracion_min"] < 2).sum())
print("Trips under 3 minutes:", (trips["Duracion_min"] < 3).sum())

print("Trips over 24 hours (1440 min):", (trips["Duracion_min"] > 1440).sum())
print("Trips over 8 hours (720 min):", (trips["Duracion_min"] > 480).sum())
print("Trips over 4 hours (240 min):", (trips["Duracion_min"] > 240).sum())

Trips with 0 minutes duration: 2780
Trips under 1 minute: 117226
Trips under 2 minutes: 171000
Trips under 3 minutes: 312544
Trips over 24 hours (1440 min): 6
Trips over 8 hours (720 min): 116
Trips over 4 hours (240 min): 273


## 6b. Check whether short trips are followed by a retry

For each user, sort trips chronologically and measure the gap between a trip's end and that same user's *next* trip start. If sub-1-minute trips are frequently followed by another trip within a few minutes, that's evidence of a docking retry pattern rather than random noise. Informing whether these rows should be flagged as invalid or treated as a real (if brief) part of the user's journey.

In [15]:
trips_sorted = trips.sort_values(["Usuario_Id", "Inicio_del_viaje"]).reset_index(drop=True)

trips_sorted["next_start"] = trips_sorted.groupby("Usuario_Id")["Inicio_del_viaje"].shift(-1)
trips_sorted["gap_to_next_min"] = (trips_sorted["next_start"] - trips_sorted["Fin_del_viaje"]).dt.total_seconds() / 60

short_trips = trips_sorted[trips_sorted["Duracion_min"] < 1]

print("Short trips (<1 min):", len(short_trips))
print()
print(short_trips["gap_to_next_min"].describe())
print()
print("Followed by another trip within 5 min:", (short_trips["gap_to_next_min"] <= 5).sum())
print("Followed by another trip within 10 min:", (short_trips["gap_to_next_min"] <= 10).sum())
print("No following trip at all (NaN, last trip for that user):", short_trips["gap_to_next_min"].isna().sum())

Short trips (<1 min): 117226

count    116697.000000
mean        713.665918
std        6418.460533
min          -0.650000
25%           0.283333
50%           0.500000
75%           4.250000
max      423302.733333
Name: gap_to_next_min, dtype: float64

Followed by another trip within 5 min: 88192
Followed by another trip within 10 min: 90743
No following trip at all (NaN, last trip for that user): 529


## 7. Flag invalid trips

The official MiBici terms of service specifies:
- Maximum continuous use is 8 hours (480 min) per the rental terms; exceeding it twice is grounds for account termination.


Cutoffs: duration < 1 minute or > 480 minutes (8 hours). The lower bound is evidence-based — 75.2% of sub-1-minute trips are followed by another trip from the same user within 5 minutes (median gap: 30 seconds), indicating a checkout-retry pattern rather than genuine short rides. The upper bound comes directly from MiBici's rental contract, which sets 8 continuous hours as the maximum permitted u.y.

Rows are flagged, not removed, preserving the full dataset for transparency. Downstream analysis can filter on `is_valid_duration` as needed.

In [17]:
trips["is_valid_duration"] = trips["Duracion_min"].between(1, 480)

print(trips["is_valid_duration"].value_counts())
print()
print(f"{(~trips['is_valid_duration']).sum()} trips flagged as invalid duration "
      f"({(~trips['is_valid_duration']).mean()*100:.2f}% of total)")

is_valid_duration
True     4414690
False     117342
Name: count, dtype: int64

117342 trips flagged as invalid duration (2.59% of total)


## 8. Investigate Año_de_nacimiento before setting validity rules

Checking for missing values and the actual range of birth years present, since we already know this column has some nulls (it was cast to float64 in staging because of them).

In [19]:
print("Missing values:", trips["Año_de_nacimiento"].isna().sum())
print(f"({trips['Año_de_nacimiento'].isna().mean()*100:.2f}% of total)")
print()
print(trips["Año_de_nacimiento"].describe())

Missing values: 1105
(0.02% of total)

count    4.530927e+06
mean     1.990418e+03
std      1.417518e+01
min      1.990000e+02
25%      1.985000e+03
50%      1.993000e+03
75%      1.999000e+03
max      2.023000e+03
Name: Año_de_nacimiento, dtype: float64


## 8b. Check for corrupted or implausible birth years

`min` (199) and `max` (2023) both suggest data entry errors rather than real users. Checking value counts at the extremes to see whether these are isolated outliers or a repeated pattern (e.g., a placeholder value used for "unknown").

In [21]:
print("Years below 1000 (likely corrupted, missing a digit):")
print(trips[trips["Año_de_nacimiento"] < 1000]["Año_de_nacimiento"].value_counts())
print()

print("Years between 1000 and 1930 (implies age >95 in 2025):")
print(len(trips[(trips["Año_de_nacimiento"] >= 1000) & (trips["Año_de_nacimiento"] < 1930)]))
print()

print("Years after 2015 (implies age <10 in 2025):")
print(trips[trips["Año_de_nacimiento"] > 2015]["Año_de_nacimiento"].value_counts())
print()

Years below 1000 (likely corrupted, missing a digit):
Año_de_nacimiento
199.0    113
Name: count, dtype: int64

Years between 1000 and 1930 (implies age >95 in 2025):
30

Years after 2015 (implies age <10 in 2025):
Año_de_nacimiento
2023.0    398
Name: count, dtype: int64



## 9. Flag invalid birth years

The official MiBici terms of service specifies:
- Minimum subscriber age is 18 (16-17 permitted only with guardian co-signature).

Cutoff: valid if `Año_de_nacimiento` is between 1930 and 2009 (implies age ~16-95 in 2025). This range was chosen after finding two repeated placeholder-like values (199 appearing 113 times, 2023 appearing 398 times) rather than random corruption, plus 30 scattered implausible entries below 1930. Missing values fall outside this range automatically and are captured by the same flag, no separate null-handling needed. As with duration, rows are flagged, not removed.

In [23]:
trips["is_valid_birth_year"] = trips["Año_de_nacimiento"].between(1930, 2009)

print(trips["is_valid_birth_year"].value_counts())
print()
print(f"{(~trips['is_valid_birth_year']).sum()} trips flagged as invalid birth year "
      f"({(~trips['is_valid_birth_year']).mean()*100:.2f}% of total)")

is_valid_birth_year
True     4530386
False       1646
Name: count, dtype: int64

1646 trips flagged as invalid birth year (0.04% of total)


## 10. Check Genero category consistency

Looking at every distinct value present, including exact casing and whitespace, since inconsistent categories (e.g., "M" vs "m" vs "M ") would silently break any grouping or chart later if not caught here.

In [25]:
print(trips["Genero"].value_counts(dropna=False))

Genero
M       3246143
F       1245100
None      40789
Name: count, dtype: int64


## 11. Flag missing Genero values

Only two valid categories present (M/F), no casing or formatting inconsistencies found. The only issue is 40,789 missing values (0.9%), flagged for consistency with the other quality checks. Rows are kept, not removed.

In [27]:
trips["is_valid_genero"] = trips["Genero"].isin(["M", "F"])

print(trips["is_valid_genero"].value_counts())

is_valid_genero
True     4491243
False      40789
Name: count, dtype: int64


## 12. Referential integrity; station IDs

Every `Origen_Id` and `Destino_Id` in trips should exist in the stations table. A mismatch here would mean some trips can't be joined to station coordinates later, important to catch now, before the modeling phase, since it directly affects the distance calculation and any map visuals.

In [29]:
valid_station_ids = set(stations["id"])

trips["origen_id_valid"] = trips["Origen_Id"].isin(valid_station_ids)
trips["destino_id_valid"] = trips["Destino_Id"].isin(valid_station_ids)

print("Invalid Origen_Id:", (~trips["origen_id_valid"]).sum())
print("Invalid Destino_Id:", (~trips["destino_id_valid"]).sum())
print()
print("Origen_Id values not found in stations (if any):")
print(trips.loc[~trips["origen_id_valid"], "Origen_Id"].value_counts().head(20))
print()
print("Destino_Id values not found in stations (if any):")
print(trips.loc[~trips["destino_id_valid"], "Destino_Id"].value_counts().head(20))

Invalid Origen_Id: 0
Invalid Destino_Id: 0

Origen_Id values not found in stations (if any):
Series([], Name: count, dtype: int64)

Destino_Id values not found in stations (if any):
Series([], Name: count, dtype: int64)


## 13. Cleaning summary

Consolidated view of every quality flag applied. No rows were removed at any point, every check adds a boolean column so downstream analysis (or Power BI) can filter as needed, while the full dataset remains intact and auditable.

In [46]:
summary = pd.DataFrame({
    "check": [
        "Valid duration (1-480 min)",
        "Valid birth year (1930-2009)",
        "Valid Genero (M/F)",
        "Valid Origen_Id",
        "Valid Destino_Id",
    ],
    "invalid_count": [
        (~trips["is_valid_duration"]).sum(),
        (~trips["is_valid_birth_year"]).sum(),
        (~trips["is_valid_genero"]).sum(),
        (~trips["origen_id_valid"]).sum(),
        (~trips["destino_id_valid"]).sum(),
    ],
})
summary["invalid_pct"] = (summary["invalid_count"] / len(trips) * 100).round(3)

summary

,check,invalid_count,invalid_pct
0,Valid duration (1-480 min),117342,2.589
1,Valid birth year (1930-2009),1646,0.036
2,Valid Genero (M/F),40789,0.900
3,Valid Origen_Id,0,0.000
4,Valid Destino_Id,0,0.000


## 14. Write cleaned data to a new schema

Mirrors the same separation used in staging: raw data stays untouched in `staging`, and this flagged-but-complete version lives in a new `cleaned` schema, ready to be built into the star schema in Phase 4.

In [49]:
with engine.connect() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS cleaned;"))
    conn.commit()

Write the cleaned, flagged trips table and the stations table into the new schema.

In [53]:
trips.to_sql("trips", con=engine, schema="cleaned", if_exists="replace", index=False)
stations.to_sql("stations", con=engine, schema="cleaned", if_exists="replace", index=False)

print(f"Loaded {trips.shape[0]} rows into cleaned.trips")
print(f"Loaded {stations.shape[0]} rows into cleaned.stations")

Loaded 4532032 rows into cleaned.trips
Loaded 484 rows into cleaned.stations


## Note: minor discrepancy vs. official MiBici dashboard

Cross-checking top station trip counts against MiBici's public 2025 dashboard showed small differences (0-2 trips per station, under 0.003% of each station's total). No duplicate Viaje_Id rows were found in this dataset. The inconsistent direction of the differences (sometimes higher, sometimes lower than the official count) suggests isolated edge cases rather than a systematic issue in this pipeline, likely differences between this static CSV export and any corrections made in MiBici's live system afterward. Documented as a known, negligible variance rather than a data quality concern.